# Adaptive SPE10 Darcy: published indicators and classical RT2 comparison

The layer-36 problem follows Barrenechea et al. (2026), §6.2, [DOI 10.1137/24M1673073](https://doi.org/10.1137/24M1673073). The principal campaign retains local P2 pressure, P0 normal-flux traces and four local triangles per macroelement. The indicator uses the printed equations (5.3)–(5.7), without inverse-material or ellipticity factors. The remeshing uses the documented FreeFEM residual metric with coefficient one and BAMG 4.13. A public note by Martins identifies that routine; its modified wrapper and historical connectivity remain unavailable, so the mesh sequence is not claimed identical. The separately refined classical RT2/P2 field is a numerical reference, not an exact solution.

Permeability is the unscaled horizontal SPE10 tensor in mD and coordinates are ft. The article and public remeshing note do not state a global material normalization; a pressure profile alone cannot identify it. Multiplying K by a constant preserves this pressure solution but changes the relative weights of the three printed indicator components.


The main equations execute from source and state their current discretization. The original archived comparisons below execute with `--historical` and require their exact attributed payloads. `--study` runs the complete public current acquisition declared in the catalogue, with its own output provenance.


In [ ]:
from pathlib import Path
import sys

# Locate the checkout from the repository root or any notebook subdirectory.
ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file()
     and (path / "src" / "pymhm").is_dir()
     and (path / "notebooks").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Run this notebook from the PyMHM checkout.")
root = ROOT
for directory in (ROOT, ROOT / "examples"):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

import os
HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
import json
from pathlib import Path
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm import TriangleMesh
from pymhm.estimators.darcy_local import estimate_darcy_local_refinement


## Execute one adaptive step on the full SPE10 layer

The immediate control uses the versioned horizontal layer-36 tensor, the entire
$1200\times2200$ ft domain and the original pressure/impermeable-wall data.
Local P2 pressure, P0 normal Darcy-flux traces and four local triangles per macroelement
match the published approximation spaces. The initial macro mesh has eight
triangles. The first solution supplies the literal printed indicator to
Dörfler marking ($\theta=1/2$), then conforming red-green refinement transfers
oriented trace and boundary data before a second solve.

This explicit adaptive step is a current marking policy; the historical BAMG
sequence below uses a different documented remeshing policy. The printed
heterogeneous indicator has no asserted coefficient-independent constant-one
energy bound. Its first reconstruction takes approximately one minute on a
single CPU thread because quadrature resolves the permeability pixels.

## Declare the local and global equations

The application writes local A, B and C blocks with the shared FEM kernels,
then supplies them to the generic assembler. Primal coefficients satisfy

$$
\begin{aligned}
a_T(p,v)&=(K\nabla p,\nabla v)_T,\\
L_T(v)&=(f,v)_T+\sum_j\alpha_jv(x_j),\\
B_T\lambda&=\langle\lambda,v\rangle_{\partial T},\\
C_Tp&=-\langle p,\mu\rangle_{\partial T}.
\end{aligned}
$$

Signs follow the fixed macroface normals. RT0 instead uses flux, pressure and
private boundary-pressure coefficients, including the local normal-flux
constraint. The global `Equation` supplies the signed Dirichlet moments and
retains each constant pressure mode. Pure Neumann data add the physical pressure
integral through an explicit constraint. `define_darcy` returns equations;
`assemble`, the global solve and field interpretation are separate stages.


In [ ]:
from pymhm import assemble, FaceSpace, SkeletonSpace
from pymhm.estimators.darcy_energy import estimate_darcy_indicator
from pymhm.adaptivity.darcy import mark_dorfler
from pymhm.meshes.refinement import refine_triangles, transfer_skeleton
from examples.formulations.darcy import define_darcy, pressure_constraints, recover_darcy
from examples.solve_spe10 import load_layer, pressure_boundary
from examples.spe10_adaptive import mesh_rectangle, natural_faces

mesh = mesh_rectangle(2, 2)
skeleton = SkeletonSpace(mesh)
material = load_layer(36)
solutions = []
with threadpool_limits(1):
    for level in range(2):
        natural = natural_faces(mesh)
        definition = define_darcy(
            mesh, permeability=material, source=0.0, dirichlet=pressure_boundary,
            neumann=natural, skeleton=skeleton, degree=2, local_refinement=2,
            quadrature_order=6,
        )
        system = assemble(definition.problem)
        solution = recover_darcy(
            definition, system,
            system.solve(constraints=pressure_constraints(definition, system)),
        )
        solutions.append(solution)
        assert solution.hybrid.residual < 1e-10
        assert np.max(np.abs(solution.conservation_residuals())) < 1e-10
        print({"level": level, "macro_cells": len(mesh.cells),
               "residual": solution.hybrid.residual,
               "macro_balance_max": float(np.max(np.abs(solution.conservation_residuals())))})
        if level == 0:
            indicator = estimate_darcy_indicator(
                solution, convention="published", degree=2,
                dirichlet=pressure_boundary, neumann=natural, quadrature_order=6,
            )
            marked = mark_dorfler(indicator.local_squared, 0.5)
            print({"published_indicator": indicator.total,
                   "marked_macro_cells": int(np.count_nonzero(marked))})
            refinement = refine_triangles(mesh, marked)
            skeleton = transfer_skeleton(skeleton, refinement, new_face=FaceSpace.uniform(0))
            mesh = refinement.mesh
assert len(solutions[1].skeleton.mesh.cells) > len(solutions[0].skeleton.mesh.cells)

### Independent same-data conforming refinement controls

The three current Q1 references below retain the reservoir, operator, pixel
material, source and boundary data. Material-cut quadrature integrates the
permeability discontinuities. Their measured physical increments check their
own refinement; these immediate meshes are still sensitive to unresolved
channels and are not accepted as a converged baseline. The full classical
RT2/P2 comparison remains the separately acquired `--study` reference.

In [ ]:
from pymhm import CartesianMacroMesh
from examples.formulations.conforming import conforming_quadrilateral
from examples.periodic_norms import difference

references = []
with threadpool_limits(1):
    for nx, ny in ((12, 22), (24, 44), (48, 88)):
        reference_mesh = CartesianMacroMesh(nx, ny, (0, 1200, 0, 2200))
        reference_natural = {
            int(face): 0.0 for face in reference_mesh.boundary_faces
            if abs(reference_mesh.normals[face, 0]) > 0.5
        }
        references.append(conforming_quadrilateral(
            reference_mesh, degree=1, permeability=material,
            dirichlet=pressure_boundary, neumann=reference_natural,
            quadrature_order=3,
        ))
    for fine, coarse in zip(references[1:], references[:-1]):
        print(difference(fine, (coarse,)))
print("Refinement sensitivity controls; these levels are not a converged baseline.")

## Independently refined reference

RT2 means the mathematical 15-dimensional local Raviart–Thomas flux space; pressure is discontinuous P2. Counts include both fields. The last 240×880 to 480×1760 refinement changes the physical flux by 3.8672% in L2 and 3.0152% in the inverse-permeability energy norm. Independent polynomial quadrature orders agree for those integrated polynomial-field comparisons. Coordinates remain ft and permeability remains mD; no viscosity conversion is added.


In [ ]:
if HISTORICAL_REPLAY:
    base = root / "examples/results/spe10-adaptive"
    references = json.loads((base / "references-aligned.json").read_text())
    for row in references:
        print(row["nx"], row["ny"], row["dofs"], row["inflow"], row["previous_flux_relative_difference"])
    print(json.loads((base / "reference-energy-final.json").read_text()))


## Adaptive fields and integrated comparisons

The raw flux, canonical RT2 reconstruction and classical mixed RT2 field are distinct. Macro balances use skeletal fluxes; reconstruction equilibrium uses continuous local P2 tests. The nested local energy difference is measured with unchanged skeletal data, but does not alter the four-triangle local spaces. The printed indicator is not asserted to be a coefficient-independent energy upper bound for heterogeneous permeability. Material cuts change integration only. All spatial panels show the actual final macro mesh, including its overlay on the reference.

Figure 9 names the raw pressure u_Hh, which is the field compared here. The Oswald potential is used only in the indicator. The sequence has seven states and reaches 4,783 macrotriangles / 12,001 global unknowns, whereas the article reports 3,786 / 9,559. Agreement of a diagonal profile is distinguished from integrated flux accuracy.


In [ ]:
if HISTORICAL_REPLAY:
    records = json.loads((base / "published/adaptive.json").read_text())
    for row in records:
        print(row["level"], row["macro_triangles"], row["local_refinement"],
              row["estimator"], row["local_indicator"], row["inflow"], row["decision"],
              row["minimum_macro_angle_degrees"])
    comparisons = json.loads((base / "published/comparison-order4.json").read_text())
    for row in comparisons:
        print(row)
    mesh_pages = ["meshes" if page == 0 else f"meshes-{page+1}" for page in range((len(records)+3)//4)]
    for name in ("refinement", "resolution-diagnostics", *mesh_pages, "pressure", "flux-magnitude", "flux-components", "reference-refinement", "diagonal-profile"):
        display(Image(filename=str(root / "docs/figures/spe10-adaptive" / f"{name}.png")))
    display(Image(filename=str(root / "docs/figures/reservoir-papers/l09-figure-9.png")))


## Profile and skeletal trace comparisons

The blue initial and green adapted curves in Figure 9 are digitized independently of the numerical solves. Their RMS differences are sample statistics, with horizontal and vertical raster uncertainty. The normal-trace diagnostic uses arclength L2 on interior faces counted once; its best P0 projection is an approximation-space diagnostic, not a volume error bound. No enriched trace solves are substituted for the declared P0 campaign.


In [ ]:
if HISTORICAL_REPLAY:
    profile = json.loads((base / "published-profile-mhm.json").read_text())
    for curve in profile["curves"]:
        print({key: value for key, value in curve.items() if key != "samples"})
    trace = json.loads((base / "published/trace-comparison.json").read_text())
    for row in trace["rows"]:
        print(row["level"], row["segments"], row["relative_difference"], row.get("actual_relative_difference"))
    coarse_rule = json.loads((base / "published/comparison-level6-order4.json").read_text())[0]
    fine_rule = json.loads((base / "published/comparison-level6-order5.json").read_text())[0]
    keys = [key for key in coarse_rule if key.endswith(("_l2", "_norm", "_difference"))]
    relative = [abs(coarse_rule[key] - fine_rule[key]) / max(abs(fine_rule[key]), 1e-30) for key in keys]
    assert max(relative) < 1e-10
    print({"quadrature_4_vs_5_max_relative_difference": max(relative)})


## Solved local, trace and material-resolution controls

These nine solves keep the final 4,783 macrotriangles, physical data, P2 local pressure and RT2 reconstruction. The published-space baseline retains four local triangles and one P0 trace. The additional controls change local resolution, the number of independent P0 segments, or material fitting explicitly.

The material-fitted r4/s4 control has reconstructed-flux differences of 22.87% in L2 and 21.50% in the inverse-permeability norm. Those remain numerical-reference differences: the final classical RT2 refinement changes its flux by 3.87%. H(div) conformity and conserved moments do not guarantee that reconstruction reduces either error.

Raw-flux Galerkin identities check local and skeletal changes independently. The energy difference has opposite signs for these two enrichments; none of the reported differences is an orthogonal decomposition relative to the classical reference.

The joint material-fitted r8/s8 control has reconstructed-flux differences of 13.29238% in L2 and 14.69486% in the inverse-permeability norm. It changes both local and skeletal resolution, so the improvement is not attributed separately to either change. Its norm quadratures 4/5 agree within 5.06e-11 relatively. The thin fitted triangles and explicit extended local residual refinement are recorded in the case documentation.

All nine controls and the seven adaptive states use the same 480×1760 classical RT2 field as the norm denominator, with its checksum recorded in every comparison.


In [ ]:
if HISTORICAL_REPLAY:
    controls = json.loads((base / "resolution/comparison.json").read_text())
    for row in controls:
        norm = row["norms"]
        print({"r": row["local_refinement"], "segments": row["trace_segments"],
               "material_fitted": row["material_fitted"],
               "pressure_percent": 100 * norm["pressure_relative_difference"],
               "raw_flux_percent": 100 * norm["raw_flux_relative_difference"],
               "reconstructed_flux_percent": 100 * norm["reconstructed_flux_relative_difference"],
               "reconstructed_inverse_K_percent": 100 * norm["reconstructed_flux_energy_relative_difference"]})
    for name in ("independent-resolution", "material-fitting", "flux-components", "material-flux-components", "joint-flux-components"):
        display(Image(filename=str(root / f"docs/figures/spe10-adaptive/resolution/{name}.png")))
